In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🚩 Feature Flags & Experimentation Guide

**Deploying safely with feature toggles, A/B testing, and gradual rollouts**

This guide covers:
- Feature flag patterns
- Toggle management
- A/B testing frameworks
- Experimentation platforms
- Gradual rollouts and canary deployments
- Feature flag infrastructure
- Analytics and metrics collection
- Rollback strategies
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Feature Flag Management

### Feature Flag Core
```python
from dataclasses import dataclass
from typing import Dict, List, Optional, Callable
from datetime import datetime
from enum import Enum

class FlagType(Enum):
    """Types of feature flags"""
    RELEASE = "release"  # Gradually roll out features
    EXPERIMENTAL = "experimental"  # A/B testing
    OPS = "ops"  # Operational toggles
    PERMISSION = "permission"  # User permissions

@dataclass
class FeatureFlag:
    """Feature flag definition"""
    flag_key: str
    flag_name: str
    flag_type: FlagType
    description: str
    enabled: bool
    owner_team: str
    created_at: datetime
    expires_at: Optional[datetime] = None
    variations: Dict[str, Dict] = None  # For experiments
    
    def is_active(self) -> bool:
        """Check if flag is active"""
        if self.expires_at and datetime.now() > self.expires_at:
            return False
        
        return self.enabled
    
    def is_expired(self) -> bool:
        """Check if flag has expired"""
        return self.expires_at and datetime.now() > self.expires_at

class FeatureFlagRule:
    """Rule for flag evaluation"""
    
    def __init__(self, rule_id: str, condition: Callable[[Dict], bool]):
        self.rule_id = rule_id
        self.condition = condition
        self.enabled = True
    
    def matches(self, context: Dict) -> bool:
        """Check if rule matches context"""
        if not self.enabled:
            return False
        
        return self.condition(context)

class FeatureFlagEvaluator:
    """Evaluate feature flags"""
    
    def __init__(self):
        self.flags: Dict[str, FeatureFlag] = {}
        self.rules: Dict[str, List[FeatureFlagRule]] = {}
        self.cache: Dict[str, Dict] = {}
    
    def register_flag(self, flag: FeatureFlag):
        """Register feature flag"""
        self.flags[flag.flag_key] = flag
    
    def add_rule(self, flag_key: str, rule: FeatureFlagRule):
        """Add evaluation rule"""
        if flag_key not in self.rules:
            self.rules[flag_key] = []
        
        self.rules[flag_key].append(rule)
    
    def is_enabled(self, flag_key: str, context: Dict = None) -> bool:
        """Evaluate if flag is enabled for context"""
        
        context = context or {}
        cache_key = f"{flag_key}:{hash(str(context))}"
        
        if cache_key in self.cache:
            return self.cache[cache_key].get('enabled', False)
        
        flag = self.flags.get(flag_key)
        
        if not flag or not flag.is_active():
            return False
        
        # Check rules
        rules = self.rules.get(flag_key, [])
        
        for rule in rules:
            if rule.matches(context):
                result = True
                self.cache[cache_key] = {'enabled': result}
                return result
        
        result = flag.enabled
        self.cache[cache_key] = {'enabled': result}
        return result
    
    def get_variation(self, flag_key: str, user_id: str) -> Optional[str]:
        """Get variation for user (for A/B tests)"""
        
        flag = self.flags.get(flag_key)
        
        if not flag or not flag.variations:
            return None
        
        # Deterministic bucketing
        hash_value = hash(f"{user_id}_{flag_key}") % 100
        
        cumulative = 0
        for variation, config in flag.variations.items():
            traffic_percent = config.get('traffic_percentage', 0)
            
            if cumulative <= hash_value < cumulative + traffic_percent:
                return variation
            
            cumulative += traffic_percent
        
        return None

class FeatureFlagManager:
    """Manage feature flags"""
    
    def __init__(self):
        self.evaluator = FeatureFlagEvaluator()
        self.metrics: Dict[str, Dict] = {}
    
    def enable_flag(self, flag_key: str) -> bool:
        """Enable flag"""
        if flag_key in self.evaluator.flags:
            self.evaluator.flags[flag_key].enabled = True
            return True
        
        return False
    
    def disable_flag(self, flag_key: str) -> bool:
        """Disable flag"""
        if flag_key in self.evaluator.flags:
            self.evaluator.flags[flag_key].enabled = False
            return True
        
        return False
    
    def set_rollout_percentage(self, flag_key: str, percentage: int) -> bool:
        """Set rollout percentage (0-100)"""
        
        flag = self.evaluator.flags.get(flag_key)
        
        if not flag:
            return False
        
        # Create rule for percentage rollout
        def percent_rule(context: Dict) -> bool:
            user_id = context.get('user_id', '')
            user_hash = hash(user_id) % 100
            return user_hash < percentage
        
        rule = FeatureFlagRule(f"rollout_{percentage}", percent_rule)
        self.evaluator.add_rule(flag_key, rule)
        
        return True
    
    def record_exposure(self, flag_key: str, user_id: str, variation: str):
        """Record flag exposure for analytics"""
        
        if flag_key not in self.metrics:
            self.metrics[flag_key] = {'exposures': 0, 'variations': {}}
        
        self.metrics[flag_key]['exposures'] += 1
        
        if variation not in self.metrics[flag_key]['variations']:
            self.metrics[flag_key]['variations'][variation] = 0
        
        self.metrics[flag_key]['variations'][variation] += 1
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. A/B Testing & Experiments

### Experiment Framework
```python
class ExperimentStatus(Enum):
    """Experiment lifecycle"""
    DRAFT = "draft"
    ACTIVE = "active"
    PAUSED = "paused"
    COMPLETED = "completed"
    CANCELLED = "cancelled"

@dataclass
class Experiment:
    """A/B test experiment"""
    experiment_id: str
    name: str
    description: str
    hypothesis: str
    start_date: datetime
    end_date: Optional[datetime]
    status: ExperimentStatus
    variants: Dict[str, Dict]  # variant_id -> config
    target_audience: Dict  # Targeting rules
    sample_size: int
    success_metrics: List[str]
    
    def is_running(self) -> bool:
        """Check if experiment is running"""
        return self.status == ExperimentStatus.ACTIVE and datetime.now() <= self.end_date

class ExperimentRunner:
    """Execute and track experiments"""
    
    def __init__(self):
        self.experiments: Dict[str, Experiment] = {}
        self.results: Dict[str, Dict] = {}
    
    def start_experiment(self, experiment: Experiment) -> bool:
        """Start new experiment"""
        
        if experiment.experiment_id in self.experiments:
            return False
        
        experiment.status = ExperimentStatus.ACTIVE
        self.experiments[experiment.experiment_id] = experiment
        
        return True
    
    def assign_variant(self, experiment_id: str, user_id: str) -> Optional[str]:
        """Assign user to variant"""
        
        experiment = self.experiments.get(experiment_id)
        
        if not experiment or not experiment.is_running():
            return None
        
        # Deterministic assignment
        hash_value = hash(f"{user_id}_{experiment_id}") % 100
        
        cumulative = 0
        for variant_id, variant_config in experiment.variants.items():
            traffic_pct = variant_config.get('traffic_percentage', 0)
            
            if cumulative <= hash_value < cumulative + traffic_pct:
                return variant_id
            
            cumulative += traffic_pct
        
        return None
    
    def record_metric(self, experiment_id: str, user_id: str, metric_name: str, value: float):
        """Record metric for experiment"""
        
        if experiment_id not in self.results:
            self.results[experiment_id] = {}
        
        if metric_name not in self.results[experiment_id]:
            self.results[experiment_id][metric_name] = {}
        
        self.results[experiment_id][metric_name][user_id] = value
    
    def calculate_statistics(self, experiment_id: str, metric_name: str) -> Dict:
        """Calculate statistical results"""
        
        experiment = self.experiments.get(experiment_id)
        metrics = self.results.get(experiment_id, {}).get(metric_name, {})
        
        if not metrics:
            return {}
        
        values = list(metrics.values())
        
        mean = sum(values) / len(values)
        variance = sum((v - mean) ** 2 for v in values) / len(values)
        std_dev = variance ** 0.5
        
        return {
            'metric': metric_name,
            'count': len(values),
            'mean': mean,
            'std_dev': std_dev,
            'min': min(values),
            'max': max(values)
        }
    
    def end_experiment(self, experiment_id: str, winner: Optional[str] = None) -> bool:
        """End experiment and declare winner"""
        
        experiment = self.experiments.get(experiment_id)
        
        if not experiment:
            return False
        
        experiment.status = ExperimentStatus.COMPLETED
        experiment.end_date = datetime.now()
        
        if winner:
            print(f"✅ Experiment {experiment_id} winner: {winner}")
        
        return True
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Gradual Rollouts & Canary Deployments

### Rollout Strategy
```python
class RolloutStage:
    """Stage in gradual rollout"""
    
    def __init__(self, stage_id: str, percentage: int, duration_hours: int):
        self.stage_id = stage_id
        self.percentage = percentage
        self.duration_hours = duration_hours
        self.start_time: Optional[datetime] = None
        self.completed = False
    
    def start(self):
        """Start rollout stage"""
        self.start_time = datetime.now()
    
    def is_complete(self) -> bool:
        """Check if stage is complete"""
        
        if not self.start_time:
            return False
        
        elapsed_hours = (datetime.now() - self.start_time).total_seconds() / 3600
        
        return elapsed_hours >= self.duration_hours

class GradualRolloutManager:
    """Manage gradual feature rollout"""
    
    def __init__(self, feature_id: str):
        self.feature_id = feature_id
        self.stages: List[RolloutStage] = []
        self.current_stage_index = 0
        self.metrics: Dict[str, float] = {}
    
    def add_stage(self, percentage: int, duration_hours: int) -> bool:
        """Add rollout stage"""
        
        stage = RolloutStage(f"stage_{len(self.stages)}", percentage, duration_hours)
        self.stages.append(stage)
        
        return True
    
    def start_rollout(self) -> bool:
        """Start rollout"""
        
        if not self.stages:
            return False
        
        self.stages[0].start()
        return True
    
    def advance_stage(self) -> bool:
        """Move to next stage if current is complete"""
        
        if self.current_stage_index >= len(self.stages):
            return False
        
        current = self.stages[self.current_stage_index]
        
        if not current.is_complete():
            return False
        
        current.completed = True
        
        if self.current_stage_index < len(self.stages) - 1:
            self.current_stage_index += 1
            self.stages[self.current_stage_index].start()
            print(f"📊 Advancing to stage {self.current_stage_index}")
            return True
        
        return False
    
    def get_rollout_percentage(self) -> int:
        """Get current rollout percentage"""
        
        if self.current_stage_index < len(self.stages):
            return self.stages[self.current_stage_index].percentage
        
        return 100  # Fully rolled out
    
    def should_rollback(self, error_rate: float, threshold: float = 0.05) -> bool:
        """Determine if rollback needed"""
        
        return error_rate > threshold
    
    def record_metric(self, metric_name: str, value: float):
        """Record rollout metric"""
        self.metrics[metric_name] = value
```

### Canary Deployment
```python
class CanaryDeployment:
    """Canary deployment strategy"""
    
    def __init__(self, deployment_id: str):
        self.deployment_id = deployment_id
        self.canary_percentage: int = 5  # Start with 5%
        self.max_percentage: int = 100
        self.stable_version: str = ""
        self.canary_version: str = ""
        self.health_checks: List[Callable[[Dict], bool]] = []
    
    def add_health_check(self, check: Callable[[Dict], bool]):
        """Add canary health check"""
        self.health_checks.append(check)
    
    def evaluate_canary_health(self, metrics: Dict) -> bool:
        """Evaluate if canary is healthy"""
        
        for check in self.health_checks:
            if not check(metrics):
                return False
        
        return True
    
    def promote_canary(self, percentage_increment: int = 10) -> int:
        """Increase canary traffic"""
        
        self.canary_percentage = min(
            self.canary_percentage + percentage_increment,
            self.max_percentage
        )
        
        return self.canary_percentage
    
    def rollback(self):
        """Rollback canary"""
        self.canary_percentage = 0
        print(f"🔄 Rolled back deployment {self.deployment_id}")
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Feature Flags & Experimentation Checklist

✅ **Flag Infrastructure**
- [ ] Flag service deployed
- [ ] SDK integrated
- [ ] Caching strategy
- [ ] Performance baseline
- [ ] Monitoring alerts

✅ **Feature Flag Usage**
- [ ] Flags defined and documented
- [ ] Rules configured
- [ ] Percentage rollouts set
- [ ] Expirations scheduled
- [ ] Cleanup process

✅ **A/B Testing**
- [ ] Experiment framework
- [ ] Variant assignment
- [ ] Metric tracking
- [ ] Statistical analysis
- [ ] Holdup analysis

✅ **Rollout Strategies**
- [ ] Gradual rollout plan
- [ ] Canary deployment setup
- [ ] Health checks defined
- [ ] Rollback triggers
- [ ] Communication plan

✅ **Analytics & Operations**
- [ ] Exposure logging
- [ ] Metrics collection
- [ ] Flag usage dashboard
- [ ] Incident response
- [ ] Post-deployment review
</VSCode.Cell>
```